In [ ]:
# This notebook aims to calculate the area of orthographic projections of our sample (in the .stl file) for varying sample angles, ψ.
# This is accomplished using a Monte Carlo approach.
# The projection is bounded and random coordinates generated within the area.
# The ratio of points in the sample projection to total is the ratio of areas.

using FileIO
using GeometryBasics
using MeshIO
using BenchmarkTools
using Distributions

In [2]:
# Retrieving the vertices and indices of the triangular mesh of the sample surface from the .stl file.


# Dummy sample comprising an icosphere with 320 faces.
stl = load("STL_FileExamples/Icosphere1280.stl")
vertices = GeometryBasics.coordinates(stl)
indices = GeometryBasics.faces(stl)
# Extracting the number of triangular faces used in the mesh.
const n_faces = length(indices)
# Extracting the number of vertices.
const n_vert = length(vertices)

3840

In [3]:
# Defining a function to rotate the sample about the z-axis.

"""
Rotates the sample by θ degrees. This is accomplished by mutating each vertex.

Parameters
----------
θ (float): Rotation angle, in degrees.
vertices (vector with 3-vector elements with float elements): Vertices of sample, in units of .stl file.

Returns
-------
vertices (vector with 3-vector elements with float elements): Rotated vertices of sample, in units of .stl file.
"""
function rotate!(θ :: Float32, vertices :: Vector{Point{3, Float32}}) :: Vector{Point{3, Float32}}
    # Converting the angle to radians.
    θ = deg2rad(θ)
    # Calculating rotation matrix.
    R_z = Matrix{Float32}([[cos(θ), -sin(θ), 0] [sin(θ), cos(θ), 0] [0, 0, 1]])
    # Applying rotation matrix to each vertex.
    @inbounds for i in 1:n_vert
        vertices[i] = R_z * vertices[i]
    end
    return vertices
end

# @benchmark rotate!(30f0, vertices)

rotate!

In [4]:
# Projecting the sample onto the y-z plane (the plane perpendicular to the neutron beam).
# This is accomplished by simply reading off the y and z coordinates of the vertices.

p_vertices = Vector{Vector{Float32}}(undef, n_vert)
for i in 1:n_vert
    p_vertices[i] = [vertices[i][2], vertices[i][3]]
end

In [5]:
# Grouping these projected vertices by the triangles they create.

p_triangles = Vector{Vector{Vector{Float32}}}(undef, n_faces)
for i in 1:n_faces
    p_triangles[i] = [p_vertices[indices[i][1]], p_vertices[indices[i][2]], p_vertices[indices[i][3]]]
end

In [6]:
# Finding the maximum and minimum value of each projected coordinate.
# This is used to create a 2D bounding rectangle.

max_coord = [maximum(getindex.(p_vertices, 1)), maximum(getindex.(p_vertices, 2))]
min_coord = [minimum(getindex.(p_vertices, 1)), minimum(getindex.(p_vertices, 2))]
# Calculating the area of the bounding rectangle.
const rec_area = (max_coord[1] - min_coord[1]) * (max_coord[2] - min_coord[2])

4.0f0

In [7]:
# Setting the total number of random coordinates to be generated within the bounding rectangle.

const n_tot = 100000

100000

In [21]:
# Defining the function to calculate the area, in units of the .stl file squared, of the sample projected into the y-z plane.

"""
Calculates the area of the sample projected onto the y-z plane. This is the cross-sectional area visible to the neutron beam.
Adopts a Monte Carlo approach to achieve this.
Ratio of random coordinates in the sample projection to total is equal to the ratio of sample projection area to total bounding area.

Parameters
----------
min_coord (2-vector with float elements): Minimum value of y and z coordinate, in units of .stl file.
max_coord (2-vector with float elements): Maximum value of y and z coordinate, in units of .stl file.
p_triangles (n_faces - vector of 3-vectors of 2-vectors with float elements): Coordinates of projected triangles, in units of .stl file.

Returns
-------
area (float): Sample projection area, in units of .stl file squared.
"""
function area_calc(min_coord :: Vector{Float32}, max_coord :: Vector{Float32}, p_triangles :: Vector{Vector{Vector{Float32}}}) :: Float32
    # Tallying the number of random coordinates that lie within the sample.
    n_in = 0
    # Pre-calculating the uniform distribution describing the bounding rectangle.
    y_range = Uniform(min_coord[1], max_coord[1])
    z_range = Uniform(min_coord[2], max_coord[2])
    # Pre-allocating vectors.
    p_e2 = Vector{Float32}(undef, 2)
    p_e3 = Vector{Float32}(undef, 2)
    p_t = Vector{Float32}(undef, 2)
    test = Vector{Float32}(undef, 2)
    for i in 1:n_tot
        # Generating a random 2D coordinate within the pre-defined ranges.
        y = rand(y_range)
        z = rand(z_range)
        test[1] = y
        test[2] = z
        for j in 1:n_faces
            triangle = p_triangles[j]
            # In barycentric coordinates, any point in a triangle can be expressed as (1-u-v) * V1 + u * V2 + v * V3 where u, v ≥ 0 and u + v ≤ 1.
            # Calculating p_e2 = V2 - V1, p_e3 = V3 - V1, p_t = test - V1, p_det = det(p_e2, p_e3) required for the algorithm.
            @. p_e2 = triangle[2] - triangle[1]
            @. p_e3 = triangle[3] - triangle[1]
            @. p_t = test - triangle[1]
            p_det = (p_e2[1] * p_e3[2]) - (p_e2[2] * p_e3[1])
            # Analytically calculating u and v.
            u = (1 / p_det) * ((p_t[1] * p_e3[2]) - (p_t[2] * p_e3[1]))
            v = (1 / p_det) * ((p_e2[1] * p_t[2]) - (p_e2[2] * p_t[1]))
            # Checking if the point lies within the triangle.
            if v ≥ 0 && u ≥ 0 && (u + v) ≤ 1
                n_in += 1
                break
            end
        end
    end
    area = (n_in / n_tot) * rec_area
    return area
end

@benchmark area_calc(min_coord, max_coord, p_triangles)

BenchmarkTools.Trial: 3 samples with 1 evaluation per sample.
 Range (min … max):  2.320 s …    2.557 s  ┊ GC (min … max): 0.00% … 0.00%
 Time  (median):     2.370 s               ┊ GC (median):    0.00%
 Time  (mean ± σ):   2.416 s ± 124.921 ms  ┊ GC (mean ± σ):  0.00% ± 0.00%

  █           █                                            █  
  █▁▁▁▁▁▁▁▁▁▁▁█▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁█ ▁
  2.32 s         Histogram: frequency by time         2.56 s <

 Memory estimate: 256 bytes, allocs estimate: 8.

In [15]:
# Testing the area calculation with an approximate circle of radius 1.
# Should act as a π approximator.

area_calc(min_coord, max_coord, p_triangles)

3.12684